# MCH–MFT Track Matching — Training & Testing

Simple NN score (MCH, MFT candidate) pairs.  
Each MCH track forms a **group** of up to 5 candidates; the model learns to rank the true match highest within each group.
The top-ranked candidate is accepted if its score exceeds a tunable threshold.

**Stages:** TODO: amend
1. Load data with hipe4ml
2. Feature engineering
3. Train/test split (by MCH track group)
4. XGBoost LambdaRank training
5. Group-level evaluation (efficiency, purity, MRR)

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils

from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 1. Load, Format, Engineer Data

In [ ]:
df_original = Utils.get_dataframe("FwdMatchMLCandidatesFull.root")


In [ ]:
df =df_original.copy() 
importlib.reload(Utils)

In [ ]:
df = Utils.process_dataframe(df, makedummies=True)

FEATURES = [f for f in df.columns.tolist() if f not in Utils.NON_TRAINING_FEATURES] # All trainable features
FEATURES = [ # Informative KS & DT features
    "DeltaDirection",
    "CPhiPhiMFT",
    "PtMFT",
    "RelPtDiff",
    "CXXMFT",
    "DeltaPt",
    "PullTanl",
    "DCAX",
    "DCAY",
    "SameSign",
    "PullPhi",
    "DeltaR",
]


TARGET = "IsSignal"

GROUP  = "mchID"

In [ ]:
df[FEATURES].describe()

In [ ]:
FEATURES

## 2. Sanity Checks

In [ ]:
n_mch_tracks = df["mchID"].nunique()
n_positive = df["IsSignal"].sum()
candidates_per_track = df.groupby("mchID").size()

print(f"MCH tracks:          {n_mch_tracks:,}")
print(f"Total pairs:         {len(df):,}")
print(f"True matches:        {int(n_positive):,} ({100*n_positive/len(df):.1f}%)")
print(f"Candidates per track: min={candidates_per_track.min()}, "
      f"max={candidates_per_track.max()}, "
      f"mean={candidates_per_track.mean():.2f}")

# Tracks with no true match among candidates
tracks_with_match = df.groupby("mchID")["IsSignal"].max()
n_no_match = (tracks_with_match == 0).sum()
print(f"Tracks with no true match in candidates: {n_no_match:,} "
      f"({100*n_no_match/n_mch_tracks:.1f}%)")

## 4. Train / Test Split

Split is done **by MCH track group**, not by row, to avoid data leakage  
(candidates from the same MCH track must not appear in both train and test).

In [ ]:
groups   = df[GROUP].values
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42) # shuffle the data and split into train/test sets while ensuring all candidates from the same mchID are in the same set
train_idx, test_idx = next(splitter.split(df, groups=groups))

df_train = df.iloc[train_idx].copy()
df_test  = df.iloc[test_idx].copy()

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]

print(f"Train: {len(df_train):,} pairs ({df_train[GROUP].nunique():,} MCH tracks)")
print(f"Test:  {len(df_test):,} pairs ({df_test[GROUP].nunique():,} MCH tracks)")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")

In [ ]:

scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler alongside model for inference
import joblib
joblib.dump(scaler, "track_matching_scaler.pkl")
print("Scaler fitted and saved.")
print(f"Feature means (sample): {scaler.mean_[:3].round(4)}")
print(f"Feature stds  (sample): {scaler.scale_[:3].round(4)}")

## 5. Train XGBoost LambdaRank

`objective='rank:ndcg'` optimises the ranking within each group of candidates directly,  
avoiding the class-imbalance problem of a binary classifier and matching the problem structure exactly.  

In [ ]:


# ── Hyperparameters ───────────────────────────────────────────────────────────
HIDDEN_LAYERS = [64, 32, 16]
DROPOUT       = 0.3
BATCH_SIZE    = 512
N_EPOCHS      = 100
LR            = 1e-3
PATIENCE      = 10    # early stopping

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")


# ── Model definition ──────────────────────────────────────────────────────────
class TrackMatchMLP(nn.Module):
    def __init__(self, n_features: int, hidden_layers: list, dropout: float):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_layers:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))   # single logit output
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)          # (batch,)


# ── Class imbalance weight ────────────────────────────────────────────────────
neg  = (y_train == 0).sum()
pos  = (y_train == 1).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(device)
print(f"pos_weight = {pos_weight.item():.2f}  (neg={neg:,}, pos={pos:,})")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


# ── Dataloaders ───────────────────────────────────────────────────────────────
X_tr = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr = torch.tensor(y_train.values, dtype=torch.float32)
X_te = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_te = torch.tensor(y_test.values,  dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=BATCH_SIZE, shuffle=True
)

model_nn = TrackMatchMLP(
    n_features=len(FEATURES),
    hidden_layers=HIDDEN_LAYERS,
    dropout=DROPOUT,
).to(device)

optimiser = torch.optim.Adam(model_nn.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="min", patience=5, factor=0.5
)

print(f"\nModel architecture:\n{model_nn}")
n_params = sum(p.numel() for p in model_nn.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


# ── Training loop ─────────────────────────────────────────────────────────────
train_losses, test_losses = [], []
best_test_loss = np.inf
patience_counter = 0
best_state = None

for epoch in range(1, N_EPOCHS + 1):
    # -- Train
    model_nn.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        logits = model_nn(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item() * len(y_batch)
    train_loss = epoch_loss / len(y_tr)

    # -- Evaluate
    model_nn.eval()
    with torch.no_grad():
        test_logits = model_nn(X_te.to(device))
        test_loss   = criterion(test_logits, y_te.to(device)).item()

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    scheduler.step(test_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d} | Train loss: {train_loss:.5f} | "
              f"Test loss: {test_loss:.5f}")

    # -- Early stopping
    if test_loss < best_test_loss:
        best_test_loss  = test_loss
        best_state      = {k: v.cpu().clone()
                           for k, v in model_nn.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break

# Restore best weights
model_nn.load_state_dict(best_state)
print(f"\nBest test loss: {best_test_loss:.5f}")


# ── Training curve ────────────────────────────────────────────────────────────
epochs_ran = len(train_losses)
fig, axes  = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, epochs_ran+1), train_losses,
        lw=2, color="steelblue", label="Train loss")
ax.plot(range(1, epochs_ran+1), test_losses,
        lw=2, color="tomato",    label="Test loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Training curve — full")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

zoom_start = int(0.8 * epochs_ran)
ax = axes[1]
ax.plot(range(zoom_start+1, epochs_ran+1), train_losses[zoom_start:],
        lw=2, color="steelblue", label="Train loss")
ax.plot(range(zoom_start+1, epochs_ran+1), test_losses[zoom_start:],
        lw=2, color="tomato",    label="Test loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Training curve — zoomed (last 20%)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Gap diagnostic ────────────────────────────────────────────────────────────
final_gap = abs(train_losses[-1] - test_losses[-1])
print(f"Train/test loss gap at final epoch:  {final_gap:.6f}")
print(f"{'⚠ Possible overfitting' if final_gap > 0.05 else '✓ No significant overfitting detected'}")


# ── Save ──────────────────────────────────────────────────────────────────────
torch.save(best_state, "track_matching_nn.pt")
print("Model saved to track_matching_nn.pt")

## 6. Feature Importance

## 7. Group-Level Evaluation

In [ ]:
df_test = df_test.copy()

model_nn.eval()
with torch.no_grad():
    logits = model_nn(X_te.to(device)).cpu().numpy()

df_test["score"] = torch.sigmoid(torch.tensor(logits)).numpy()

METRICS = ["score"]

## 8. All matches

In [ ]:
match_groups = Utils.build_match_groups(df_test)
Utils.draw_all_features(features=METRICS, match_groups=match_groups, density=True)

## 9. Leading Match Metric Distribution Analysis

In [ ]:
importlib.reload(Utils)
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=METRICS, match_groups=match_groups_leader, density=True, per=0.0)

## 10. Metric score sweep

In [ ]:
for entry in METRICS:
    Utils.sweep_threshold_plot(df_eval= df_test, metrics_fn = Utils.inhousemetrics, title=entry +" vs Score Threshold", score_col=entry, Nsigma=3.0)

## 11. Match Assigned Analysis

In [ ]:
# Apply threshold to get final matches, then plot feature distributions for the accepted candidates - see firsthand the contamination
threshold = 0.6
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
df_leader = df_leader[(df_leader["score"] >= threshold) & (df_leader["MatchLabel"] != 8)].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=FEATURES, match_groups=match_groups_leader, density=True)

## 12. Featurewise Metric breakdown

In [ ]:
importlib.reload(Utils)
for entry in Utils.DESIGNED_FEATURES:   
    Utils.plot_metrics_vs_feature(df=df_test,feature=entry, threshold = 0.6, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.02, trim_high=0.02, Nsigma=2.0)

## 13. Feature 

In [ ]:
from scipy.stats import ks_2samp

# ── Thresholds — adjust these ─────────────────────────────────────────────────
KS_THRESHOLD         = 0.1
IMPORTANCE_THRESHOLD = 0.01

# ── Compute KS statistic for each feature ────────────────────────────────────
signal     = df[df["IsSignal"] == 1]
background = df[df["IsSignal"] == 0]

ks_stats = {
    feature: ks_2samp(signal[feature].dropna(),
                      background[feature].dropna()).statistic
    for feature in FEATURES
}

importance = dict(zip(FEATURES, model.feature_importances_))

selection_df = pd.DataFrame({
    "ks":         ks_stats,
    "importance": importance,
}).sort_values("ks", ascending=False)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

passes = selection_df[
    (selection_df["ks"]         >= KS_THRESHOLD) &
    (selection_df["importance"] >= IMPORTANCE_THRESHOLD)
]
fails = selection_df[
    (selection_df["ks"]         <  KS_THRESHOLD) |
    (selection_df["importance"] <  IMPORTANCE_THRESHOLD)
]

ax.scatter(fails["ks"],   fails["importance"],
           alpha=0.6, color="tomato",    s=60, label="Rejected")
ax.scatter(passes["ks"],  passes["importance"],
           alpha=0.8, color="steelblue", s=60, label="Selected")

for feature, row in selection_df.iterrows():
    ax.annotate(feature, (row["ks"], row["importance"]),
                fontsize=8, textcoords="offset points", xytext=(5, 3))

ax.axvline(KS_THRESHOLD,         color="black", linestyle="--", lw=1.2,
           label=f"KS threshold ({KS_THRESHOLD})")
ax.axhline(IMPORTANCE_THRESHOLD, color="grey",  linestyle="--", lw=1.2,
           label=f"Importance threshold ({IMPORTANCE_THRESHOLD})")

ax.set_xlabel("KS statistic (univariate separability)")
ax.set_ylabel("Feature importance (model usage)")
ax.set_title("Feature selection overview")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Print selected features ───────────────────────────────────────────────────
selected = passes.index.tolist()
dropped  = fails.index.tolist()

print(f"Selected ({len(selected)}) — copy-paste ready:")
print("FEATURES = [")
for f in selected:
    print(f"    \"{f}\",")
print("]")

print(f"\nDropped ({len(dropped)}):")
print(dropped)
print(selection_df.round(4))